## Bölüm 6 — BIST Sektör Modeli

### 6.1 Yaklaşım

Amaç: BIST endeksinin altına, kendi başına endeksten daha volatil olan
ama toplamda endeksle "tutarlıymış illüzyonu" veren sektör bazlı ETF'ler
eklemek.

Kurallar:
  - Her sektörün getirisi = endeks getirisi + kendi bağımsız gürültüsü
  - Gürültünün büyüklüğü (std), endeksin REEL getirisine (enflasyon
    üstü) bağlı: endeks güçlü bir yöne gidiyorsa (güçlü boğa/ayı)
    sektörler birlikte hareket eder (düşük dağılım). Endeks reel olarak
    yatay seyrediyorsa (rotasyon yılı) sektörler arası fark açılır —
    bazıları uçarken bazıları çakılabilir.
  - Ağırlıklı ortalama endeksi BİREBİR vermez, sadece yaklaşık verir —
    bu kasıtlı, gerçekçilik için.

### 6.2 Parametreler

Sektör ağırlıkları (toplam 1.00):
  Bankacılık: %30
  Teknoloji:  %20
  İnşaat:     %20
  Sağlık:     %15
  Perakende:  %15

Dağılım (std) formülü:
  baz_std = 22 puan
  yatay_carpan = max(0.5, 1.7 - |reel_bist| / 25)
  std = baz_std * yatay_carpan

  reel_bist ≈ 0  → carpan ≈ 1.7 → std ≈ 37  (yüksek dağılım, rotasyon)
  reel_bist ≈ ±40 → carpan ≈ 0.5 → std ≈ 11  (düşük dağılım, birlikte hareket)


In [2]:
from backend.engine.bist import bist_getiri_uret, faiz_uret
from backend.engine.enflasyon import enflasyon_sim
from backend.engine.utils import normal
import random

# ============================================================
# BIST SEKTÖR MODELİ
# ============================================================

SEKTOR_AGIRLIKLARI = {
    "bankacilik": 0.30,
    "teknoloji":  0.20,
    "insaat":     0.20,
    "saglik":     0.15,
    "perakende":  0.15,
}


def sektor_dispersiyon_std(reel_bist):
    """
    Endeks reel olarak yatay seyrediyorsa (rotasyon yılı) dağılım geniş,
    endeks güçlü bir yöne gidiyorsa (birlikte hareket) dağılım dar olur.
    """
    baz_std = 22
    yatay_carpan = max(0.5, 1.7 - abs(reel_bist) / 25)
    return baz_std * yatay_carpan


def sektorleri_uret(bist_pct, reel_bist):
    """
    Her sektör için: endeks getirisi + kendi bağımsız gürültüsü.
    Ağırlıklı ortalama endeksi yaklaşık verir, birebir vermez.
    """
    std = sektor_dispersiyon_std(reel_bist)
    sonuc = {}
    for sektor in SEKTOR_AGIRLIKLARI:
        gurultu = normal(0, std, min_val=-70, max_val=90)
        sonuc[sektor] = round(bist_pct + gurultu, 1)
    return sonuc, std


### 6.3 Test — Senaryo Bazlı Doğrulama

Beş farklı senaryo: güçlü boğa, güçlü ayı, iki farklı yatay/rotasyon
durumu, ve orta şiddetli bir yıl. Beklenen:
  - Boğa/ayı senaryolarında sektörler birbirine yakın hareket etmeli
  - Yatay senaryolarda sektörler arası fark belirgin şekilde açılmalı
  - Ağırlıklı ortalama her durumda endekse yakın ama birebir aynı olmamalı


In [9]:
import random

random.seed(5)

senaryolar = [
    ("GÜÇLÜ BOĞA", 65, 55),
    ("GÜÇLÜ AYI", -45, -50),
    ("YATAY (reel ~0) #1", 12, 0.5),
    ("YATAY (reel ~0) #2", 9, -1),
    ("ORTA", 25, 12),
]

for etiket, bist_pct, reel_bist in senaryolar:
    sektorler, std = sektorleri_uret(bist_pct, reel_bist)
    agirlikli_ort = sum(sektorler[s] * SEKTOR_AGIRLIKLARI[s] for s in sektorler)

    print(f"--- {etiket} --- bist_pct=%{bist_pct} reel_bist=%{reel_bist} (dispersiyon_std={std:.1f})")
    for s, v in sektorler.items():
        print(f"  {s:<12}: %{v:>6.1f}")
    print(f"  Ağırlıklı ortalama: %{agirlikli_ort:.1f}  (gerçek endeks: %{bist_pct})")
    print()


AttributeError: 'builtin_function_or_method' object has no attribute 'seed'

### 6.4 Test — 60 Yıllık Entegre Simülasyon

Enflasyon + döviz + BIST endeksi + sektörler birlikte, gerçek oyun
akışına en yakın hâliyle.


In [ ]:
def simulate_bist_sektorler(yil_sayisi=60, seed=None):
    if seed:
        random.seed(seed)
    rejim = 0
    sakin_yil = 0
    kriz_mevcut = None
    kriz_dusus_hizi = None
    faiz = 12.0
    bist = 100.0

    sonuclar = []
    for yil in range(yil_sayisi):
        enf, rejim, sakin_yil, kriz_mevcut, kriz_dusus_hizi, durum = \
            enflasyon_sim(rejim, sakin_yil, kriz_mevcut, kriz_dusus_hizi)

        faiz, faiz_yon, faiz_mod = faiz_uret(enf, rejim, faiz)
        bist_pct = bist_getiri_uret(faiz_yon, enf, rejim)
        bist = round(bist * (1 + bist_pct / 100), 2)
        reel_bist = round(bist_pct - enf, 1)

        sektorler, std = sektorleri_uret(bist_pct, reel_bist)
        agirlikli_ort = round(sum(sektorler[s] * SEKTOR_AGIRLIKLARI[s] for s in sektorler), 1)

        sonuclar.append({
            "yil": 2025 + yil, "yas": 25 + yil,
            "bist_pct": round(bist_pct, 1), "reel_bist": reel_bist,
            "dispersiyon_std": round(std, 1),
            "sektorler": sektorler, "agirlikli_ort": agirlikli_ort,
        })
    return sonuclar


for seed in [42, 99]:
    sim = simulate_bist_sektorler(60, seed)
    print(f"\n=== SEED {seed} ===")
    for s in sim:
        if s["yas"] % 15 == 0:
            print(f"Yıl {s['yil']} Yaş {s['yas']}: endeks=%{s['bist_pct']:.1f} reel=%{s['reel_bist']:.1f} "
                  f"std={s['dispersiyon_std']:.1f} agirlikli_ort=%{s['agirlikli_ort']:.1f}")
            for sek, v in s["sektorler"].items():
                print(f"    {sek:<12}: %{v:>6.1f}")


### 6.5 Sonraki Adım

Matematik doğrulandı. Sıradaki iş:
  - `engine/bist.py`'ye (veya yeni bir `sektor.py` modülüne) taşımak
  - `simulasyon.py`'ye entegre etmek — her yıl `yil_sonucu`'na
    `sektor_getirileri` eklemek
  - Frontend'de "Borsa Sayfası" (endeks üstte, sektör ETF'leri altta,
    her biri ayrı alım-satım) — `VarlikSayfasi`'ndaki BIST kartına
    tıklanınca açılacak
  - Sektöre özel random event'ler (ayrı bir aşamada, dönemsellik
    artırılarak)
